# In-domain оценка на LLaVA-Instruct-ru holdout

GQA-ru и MMBench-ru измеряют короткие ответы и выбор варианта. Этот ноутбук дополняет их проверкой способности модели вести русскоязычный визуальный диалог и рассуждать в том же формате, на котором обучались адаптеры.

Из `LLaVA-Instruct-ru train` выбираются 100 примеров, гарантированно не входивших в исходную 12-тысячную кандидатную выборку: 50 `conversation` и 50 `complex_reasoning`. Поэтому holdout независим от F1, F2, F3 и A1. Считаются BERTScore F1, ROUGE-L и длина ответа.

In [1]:
# Все метрики ниже считаются уже установленными пакетами и функциями ноутбука.
# Для ROUGE-L используется собственная Unicode-токенизация, корректная для русского языка.

In [2]:
import gc
import json
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
import torch
import urllib3
from bert_score import score
from peft import PeftModel
from PIL import Image
from transformers import (
    AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
)

os.environ['PYTHONIOENCODING'] = 'utf-8'
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA = ROOT / 'data' / 'raw' / 'llava_instruct_ru_train.json'
TRAIN_SUBSET = ROOT / 'data' / 'processed' / 'llava_instruct_ru_first_iteration_seed42.csv'
HOLDOUT_PATH = ROOT / 'data' / 'processed' / 'llava_holdout_eval_seed2026.csv'
IMAGE_CACHE = ROOT / 'data' / 'coco_cache' / 'train2017'
RESULTS_DIR = ROOT / 'results'
BERTSCORE_MODEL = 'xlm-roberta-large'
SEED = 2026
EXAMPLES_PER_TYPE = 50
MAX_NEW_TOKENS = 128
assert torch.cuda.is_available(), 'Для генерации нужна CUDA.'
print('GPU:', torch.cuda.get_device_name(0))

GPU: NVIDIA GeForce RTX 4060


## 1. Фиксированный holdout без пересечений

In [3]:
raw_records = json.loads(RAW_DATA.read_text(encoding='utf-8'))
excluded_rows = set(pd.read_csv(TRAIN_SUBSET)['row_number'].astype(int))

def parse_record(row_number, record):
    messages = record['conversations']
    question = next(item['value'] for item in messages if item['from'] == 'human')
    answer = next(item['value'] for item in messages if item['from'] == 'gpt')
    return {
        'row_number': row_number, 'type': record['type'],
        'question': question.replace('<image>\n', '').strip(),
        'target': answer.strip(), 'image_path': record['image'],
    }

if HOLDOUT_PATH.exists():
    holdout = pd.read_csv(HOLDOUT_PATH)
else:
    pool = pd.DataFrame(
        parse_record(i, record) for i, record in enumerate(raw_records) if i not in excluded_rows
    )
    holdout = (pool.groupby('type', group_keys=False)
               .sample(n=EXAMPLES_PER_TYPE, random_state=SEED)
               .sample(frac=1, random_state=SEED).reset_index(drop=True))
    holdout.to_csv(HOLDOUT_PATH, index=False)

assert not set(holdout.row_number).intersection(excluded_rows)
display(holdout.groupby('type').size().rename('examples'))
display(holdout[['type', 'question', 'target']].head(3))

type
complex_reasoning    50
conversation         50
Name: examples, dtype: int64

,type,question,target
0,conversation,Сколько птиц на изображении?,На изображении можно увидеть четыре птицы: одн...
1,complex_reasoning,"Что может говорить о стиле и характере дома, и...","Изображение намекает на то, что дом оформлен в..."
2,complex_reasoning,Что можно сказать о социальном статусе и отнош...,"На фотографии изображены пять мужчин, одетых в..."


## 2. Изображения

Скачиваются только отсутствующие COCO-файлы. Если сеть не отвечает, включите VPN и повторите ячейку.

In [4]:
def coco_url(image_path):
    parts = Path(image_path).parts
    return f'https://images.cocodataset.org/{parts[-2]}/{parts[-1]}'

def download_image(image_path):
    target = IMAGE_CACHE / Path(image_path).name
    if target.exists():
        return True, image_path, 'cached'
    last_error = None
    for attempt in range(3):
        try:
            response = requests.get(coco_url(image_path), timeout=(15, 60), verify=False)
            response.raise_for_status()
            target.write_bytes(response.content)
            return True, image_path, 'downloaded'
        except Exception as error:
            last_error = error
            time.sleep(attempt + 1)
    return False, image_path, str(last_error)

missing = [path for path in holdout.image_path.unique() if not (IMAGE_CACHE / Path(path).name).exists()]
failed = []
print(f'Нужно скачать изображений: {len(missing)}')
if missing:
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = [executor.submit(download_image, path) for path in missing]
        for completed, future in enumerate(as_completed(futures), start=1):
            ok, image_path, status = future.result()
            if not ok:
                failed.append((image_path, status))
            if completed % 10 == 0 or completed == len(futures):
                print(f'Загружено {completed}/{len(futures)}; ошибок: {len(failed)}')
assert not failed, f'Не удалось скачать {len(failed)} изображений.'
print('Изображения holdout готовы.')

Нужно скачать изображений: 0
Изображения holdout готовы.


## 3. Модели и генерация

Каждая модель сохраняется в отдельный CSV. Повторный запуск пропускает уже завершённые варианты.

In [5]:
MODEL_REGISTRY = {
    'B0': {
        'kind': 'qwen', 'base_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'adapter': None,
    },
    'F1-fast': {
        'kind': 'qwen', 'base_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'adapter': ROOT / 'models' / 'f1_fast_qlora_adapter',
    },
    'F2': {
        'kind': 'qwen', 'base_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'adapter': ROOT / 'models' / 'f2_qwen_r16_adapter',
    },
    'F3-10k': {
        'kind': 'qwen', 'base_id': 'Qwen/Qwen2.5-VL-3B-Instruct',
        'adapter': ROOT / 'models' / 'f3_10k_qwen_r16_adapter',
    },
    'A1-InternVL3': {
        'kind': 'internvl', 'base_id': 'OpenGVLab/InternVL3-2B-hf',
        'adapter': ROOT / 'models' / 'a1_internvl3_2b_qlora_adapter',
    },
}

def result_path(model_name):
    slug = model_name.lower().replace('-', '_')
    return RESULTS_DIR / f'llava_holdout_{slug}_predictions.csv'

def load_variant(spec):
    processor_source = spec['adapter'] if spec['adapter'] else spec['base_id']
    processor_kwargs = {}
    if spec['kind'] == 'qwen' and spec['adapter'] is None:
        processor_kwargs = {'min_pixels': 256 * 28 * 28, 'max_pixels': 512 * 28 * 28}
    processor = AutoProcessor.from_pretrained(processor_source, **processor_kwargs)
    if spec['kind'] == 'internvl':
        processor.image_processor.max_patches = 1
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
    )
    model_class = (Qwen2_5_VLForConditionalGeneration
                   if spec['kind'] == 'qwen' else AutoModelForImageTextToText)
    model = model_class.from_pretrained(
        spec['base_id'], quantization_config=quantization, device_map='auto', dtype=torch.float16,
    ).eval()
    if spec['adapter']:
        model = PeftModel.from_pretrained(model, spec['adapter']).eval()
    return processor, model

def generate_answer(processor, model, image, question):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': question},
    ]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    generated = output[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

def evaluate_model(model_name, spec):
    processor, model = load_variant(spec)
    rows = []
    started = time.perf_counter()
    for i, row in holdout.iterrows():
        image = Image.open(IMAGE_CACHE / Path(row.image_path).name).convert('RGB')
        prediction = generate_answer(processor, model, image, row.question)
        rows.append({**row.to_dict(), 'model': model_name, 'prediction': prediction})
        if (i + 1) % 10 == 0:
            print(f'{model_name}: {i + 1}/{len(holdout)}')
    result = pd.DataFrame(rows)
    result.to_csv(result_path(model_name), index=False)
    print(f'{model_name}: {(time.perf_counter() - started) / 60:.1f} мин.')
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()
    return result

In [6]:
prediction_frames = []
for model_name, spec in MODEL_REGISTRY.items():
    path = result_path(model_name)
    adapter_ready = spec['adapter'] is None or (spec['adapter'] / 'adapter_model.safetensors').exists()
    if path.exists():
        print(f'{model_name}: использую сохранённые ответы.')
        frame = pd.read_csv(path)
    elif adapter_ready:
        frame = evaluate_model(model_name, spec)
    else:
        print(f'{model_name}: адаптер не найден — пропускаю.')
        continue
    prediction_frames.append(frame)
all_predictions = pd.concat(prediction_frames, ignore_index=True)
all_predictions.to_csv(RESULTS_DIR / 'llava_holdout_predictions.csv', index=False)
display(all_predictions.groupby(['model', 'type']).size().rename('examples'))

B0: использую сохранённые ответы.
F1-fast: использую сохранённые ответы.
F2: использую сохранённые ответы.
F3-10k: использую сохранённые ответы.
A1-InternVL3: использую сохранённые ответы.


model         type             
A1-InternVL3  complex_reasoning    50
              conversation         50
B0            complex_reasoning    50
              conversation         50
F1-fast       complex_reasoning    50
              conversation         50
F2            complex_reasoning    50
              conversation         50
F3-10k        complex_reasoning    50
              conversation         50
Name: examples, dtype: int64

## 4. BERTScore, ROUGE-L и длина ответа

In [7]:
precision, recall, f1 = score(
    all_predictions.prediction.fillna('').astype(str).tolist(),
    all_predictions.target.fillna('').astype(str).tolist(),
    model_type=BERTSCORE_MODEL, lang='ru', device='cpu', batch_size=8,
    rescale_with_baseline=False, verbose=True,
)
all_predictions['bertscore_f1'] = f1.cpu().numpy()

def tokenize_ru(text):
    # Стандартный rouge-score отбрасывает кириллицу. Здесь слова выделяются в Unicode.
    return re.findall(r'[^\W_]+', str(text).lower(), flags=re.UNICODE)

def rouge_l_f1(target, prediction):
    reference = tokenize_ru(target)
    candidate = tokenize_ru(prediction)
    if not reference or not candidate:
        return 0.0
    previous = [0] * (len(candidate) + 1)
    for reference_token in reference:
        current = [0]
        for j, candidate_token in enumerate(candidate, start=1):
            if reference_token == candidate_token:
                current.append(previous[j - 1] + 1)
            else:
                current.append(max(previous[j], current[-1]))
        previous = current
    lcs = previous[-1]
    precision_lcs = lcs / len(candidate)
    recall_lcs = lcs / len(reference)
    return 2 * precision_lcs * recall_lcs / (precision_lcs + recall_lcs) if lcs else 0.0

all_predictions['rouge_l'] = [
    rouge_l_f1(target, prediction)
    for target, prediction in zip(all_predictions.target, all_predictions.prediction)
]
all_predictions['prediction_words'] = all_predictions.prediction.fillna('').astype(str).str.split().str.len()
all_predictions.to_csv(RESULTS_DIR / 'llava_holdout_predictions.csv', index=False)

metrics_by_type = (all_predictions.groupby(['model', 'type'])
                   .agg(bertscore_f1=('bertscore_f1', 'mean'),
                        rouge_l=('rouge_l', 'mean'),
                        prediction_words=('prediction_words', 'mean'))
                   .mul({'bertscore_f1': 100, 'rouge_l': 100, 'prediction_words': 1})
                   .round(2))
overall_metrics = (all_predictions.groupby('model')
                   .agg(bertscore_f1=('bertscore_f1', 'mean'),
                        rouge_l=('rouge_l', 'mean'),
                        prediction_words=('prediction_words', 'mean'))
                   .mul({'bertscore_f1': 100, 'rouge_l': 100, 'prediction_words': 1})
                   .round(2).sort_values('bertscore_f1', ascending=False))
metrics_by_type.to_csv(RESULTS_DIR / 'llava_holdout_metrics_by_type.csv')
overall_metrics.to_csv(RESULTS_DIR / 'llava_holdout_metrics.csv')
display(overall_metrics)
display(metrics_by_type)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


calculating scores...
computing bert embedding.


  0%|          | 0/69 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/63 [00:00<?, ?it/s]

done in 33.98 seconds, 14.71 sentences/sec


,bertscore_f1,rouge_l,prediction_words
model,,,
F3-10k,92.31,32.53,33.46
F2,92.07,31.35,34.28
F1-fast,91.96,30.10,33.53
A1-InternVL3,91.92,31.25,35.08
B0,91.15,25.83,32.20


bertscore_f1  rouge_l  prediction_words
model        type                                                      
A1-InternVL3 complex_reasoning         89.63    20.55             58.96
             conversation              94.20    41.95             11.20
B0           complex_reasoning         88.98    16.86             52.00
             conversation              93.33    34.80             12.40
F1-fast      complex_reasoning         89.83    20.13             58.88
             conversation              94.09    40.07              8.18
F2           complex_reasoning         90.01    20.66             59.66
             conversation              94.14    42.04              8.90
F3-10k       complex_reasoning         90.24    21.32             58.78
             conversation              94.38    43.74              8.14

## 5. Качественные примеры

Показываются примеры, где F3 сильнее всего изменила семантическую близость относительно B0. Это помогает отличить реальное улучшение рассуждения от простого увеличения длины ответа.

In [8]:
if {'B0', 'F3-10k'}.issubset(set(all_predictions.model)):
    comparison = (all_predictions[all_predictions.model.isin(['B0', 'F3-10k'])]
                  .pivot(index=['row_number', 'type', 'question', 'target'],
                         columns='model', values=['prediction', 'bertscore_f1']))
    comparison['change'] = comparison[('bertscore_f1', 'F3-10k')] - comparison[('bertscore_f1', 'B0')]
    display(comparison.sort_values('change', ascending=False).head(10))

prediction  \
model                                                                                                                                                                              B0   
row_number type              question                                           target                                                                                                  
90428      conversation      Сколько разных типов вагонов поездов находится ... На железнодорожной станции находится много разл...  На станции можно увидеть несколько типов вагон...   
91202      conversation      Каким цветом является птица, сидящая на руке че... Птица белого и серого цвета.                        Птица, сидящая на руке человека, имеет серый ц...   
91054      conversation      Сколько людей стоит на вершине погрузочной плат... Группа из нескольких человек стоит на вершине п...  Извините, но я не могу увидеть людей на вершин...   
75778      conversation      Сколько самолетов находится на берегу озера?       На берегу озера находятся два самолета.             На изображении показано два самолета, стоящих ...   
105067     conversation      Сколько человек занимаются игрой в фрисби на из... На изображении несколько человек занимаются игр...  На изображении показано, что люди играют в фри...   
31828      complex_reasoning Какие факторы могут повлиять на безопасность ре... Безопасность ребенка и слона на фото может быть...  На изображении слон и человек находятся рядом ...   
46092      complex_reasoning Какие техники безопасности могут быть важны при... При катании на сноуборде или лыжах на склоне, к...  На фотографии показан человек, катящийся на сн...   
79582      conversation      Сколько мотоциклов находится на изображении?       На изображении находится большое количество мот...  Извините, но я не могу точно определить количе...   
36454      complex_reasoning Каковы потенциальные преимущества использования... Использование лошадей для транспортировки всё е...  Вот несколько преимуществ использования лошаде...   
33679      complex_reasoning Какие навыки развивает собака, занимаясь ловлей... Собака, занимающаяся ловлей летающего диска, ра...  Ловля летающего диска развивает несколько важн...   

                                                                                                                                                                                       \
model                                                                                                                                                                          F3-10k   
row_number type              question                                           target                                                                                                  
90428      conversation      Сколько разных типов вагонов поездов находится ... На железнодорожной станции находится много разл...  На железнодорожной станции находятся несколько...   
91202      conversation      Каким цветом является птица, сидящая на руке че... Птица белого и серого цвета.                                                      Птица серого цвета.   
91054      conversation      Сколько людей стоит на вершине погрузочной плат... Группа из нескольких человек стоит на вершине п...  На вершине погрузочной платформы стоит несколь...   
75778      conversation      Сколько самолетов находится на берегу озера?       На берегу озера находятся два самолета.                       На берегу озера находится два самолета.   
105067     conversation      Сколько человек занимаются игрой в фрисби на из... На изображении несколько человек занимаются игр...  На изображении несколько человек занимаются иг...   
31828      complex_reasoning Какие факторы могут повлиять на безопасность ре... Безопасность ребенка и слона на фото может быть...  Безопасность ребенка и слона на изображении мо...   
46092      complex_reasoning Какие техники безопасности могут быть важны при... При катании на сноуб

## 6. Как проверять правильность длинных ответов

BERTScore измеряет семантическую близость, а ROUGE-L — пересечение последовательностей слов. Ни одна из этих метрик не проверяет, действительно ли каждое утверждение видно на изображении. Поэтому для 10 фиксированных примеров создаётся ручная таблица: одни и те же вопросы проверяются для всех пяти моделей.

Шкала `factual_score`: **2** — главное заключение верно и существенных ошибок нет; **1** — ответ частично верен, но есть пропуск или несущественная ошибка; **0** — главное заключение неверно. В `hallucination` ставится `1`, если модель уверенно добавила отсутствующий на изображении объект, действие или признак. Полностью правильным считаем ответ с `factual_score = 2` и `hallucination = 0`. Источник истины здесь — изображение и вопрос, а не буквальное совпадение с единственным референсом.

In [9]:
manual_ids = (holdout.groupby('type', group_keys=False)
              .sample(n=5, random_state=SEED).row_number.tolist())
manual_review = (all_predictions[all_predictions.row_number.isin(manual_ids)]
                 [['row_number', 'type', 'image_path', 'question', 'target', 'model', 'prediction']]
                 .sort_values(['row_number', 'model']).reset_index(drop=True))
manual_review['factual_score_0_2'] = ''
manual_review['hallucination_0_1'] = ''
manual_review['review_comment'] = ''
manual_review.to_csv(RESULTS_DIR / 'llava_holdout_manual_review.csv', index=False)
print('Подготовлено строк для ручной проверки:', len(manual_review))
display(manual_review.head(10))

Подготовлено строк для ручной проверки: 50


,row_number,type,image_path,question,target,model,prediction,factual_score_0_2,hallucination_0_1,review_comment
0,1154,complex_reasoning,coco/train2017/000000466378.jpg,Какие действия могли предшествовать сцене на ф...,"Предположительно, на фотографии можно увидеть ...",A1-InternVL3,"На изображении видно, что на столе лежат банан...",,,
1,1154,complex_reasoning,coco/train2017/000000466378.jpg,Какие действия могли предшествовать сцене на ф...,"Предположительно, на фотографии можно увидеть ...",B0,"Изображение показывает кухонный стол, на котор...",,,
2,1154,complex_reasoning,coco/train2017/000000466378.jpg,Какие действия могли предшествовать сцене на ф...,"Предположительно, на фотографии можно увидеть ...",F1-fast,"Сцена на фотографии, где на столе лежит банан ...",,,
3,1154,complex_reasoning,coco/train2017/000000466378.jpg,Какие действия могли предшествовать сцене на ф...,"Предположительно, на фотографии можно увидеть ...",F2,"Сцена на фотографии, где на деревянном столе л...",,,
4,1154,complex_reasoning,coco/train2017/000000466378.jpg,Какие действия могли предшествовать сцене на ф...,"Предположительно, на фотографии можно увидеть ...",F3-10k,"Предположительно, перед сценой на фотографии м...",,,
5,13230,complex_reasoning,coco/train2017/000000182801.jpg,Какие меры предосторожности должен соблюдать п...,Пешеход на улице должен соблюдать меры предост...,A1-InternVL3,"Пешеход, стоящий на улице, должен быть внимате...",,,
6,13230,complex_reasoning,coco/train2017/000000182801.jpg,Какие меры предосторожности должен соблюдать п...,Пешеход на улице должен соблюдать меры предост...,B0,Пешеходу следует следовать следующим мерам пре...,,,
7,13230,complex_reasoning,coco/train2017/000000182801.jpg,Какие меры предосторожности должен соблюдать п...,Пешеход на улице должен соблюдать меры предост...,F1-fast,Пешеходу на улице следует соблюдать следующие ...,,,
8,13230,complex_reasoning,coco/train2017/000000182801.jpg,Какие меры предосторожности должен соблюдать п...,Пешеход на улице должен соблюдать меры предост...,F2,Пешеходу на улице следует соблюдать ряд мер пр...,,,
9,13230,complex_reasoning,coco/train2017/000000182801.jpg,Какие меры предосторожности должен соблюдать п...,Пешеход на улице должен соблюдать меры предост...,F3-10k,Пешеходу на улице следует соблюдать несколько ...,,,


## 7. Выводы

На согласованном LLaVA holdout лучшей стала **F3-10k**: BERTScore F1 вырос с **91,15** у B0 до **92,31**, а Unicode ROUGE-L — с **25,83** до **32,53**. На `complex_reasoning` F3 также лидирует: **90,24** BERTScore и **21,32** ROUGE-L против **88,98** и **16,86** у B0. Последовательный рост B0 → F1 → F2 → F3 по BERTScore согласуется с увеличением объёма и ёмкости обучения.

Это подтверждает специализацию адаптеров под русскоязычные развернутые ответы LLaVA, но не отменяет снижение на GQA-ru/MMBench-ru. Бенчмарки измеряют разные способности: GQA/MMBench — короткий факт и выбор варианта, этот holdout — диалоговый и объяснительный формат. Поэтому корректная формулировка — не «старые бенчмарки устарели», а «перенос на другой формат ответа ограничен».

Автоматические числа нельзя называть процентом фактически правильных ответов. Финальное утверждение о правильности и галлюцинациях следует делать после заполнения `results/llava_holdout_manual_review.csv` по приведённой шкале.